### Imports and paths

In [88]:
import pandas as pd
import numpy as np
from pathlib import Path

In [89]:
CLEANED_DATA_PATH = Path("../data/cleaned")
STAGING_DATA_PATH = Path("../data/staging")

STAGING_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Cleaned:",
    CLEANED_DATA_PATH.resolve()
)

print(
    "Staging:",
    STAGING_DATA_PATH.resolve()
)

Cleaned: C:\New folder\Hands on Projects\healthcare-operations-population-health-analytics\data\cleaned
Staging: C:\New folder\Hands on Projects\healthcare-operations-population-health-analytics\data\staging


### Load the 12 cleaned business tables

In [90]:
patients = pd.read_csv(
    CLEANED_DATA_PATH / "patients_clean.csv"
)

providers = pd.read_csv(
    CLEANED_DATA_PATH / "providers_clean.csv"
)

departments = pd.read_csv(
    CLEANED_DATA_PATH / "departments_clean.csv"
)

payers = pd.read_csv(
    CLEANED_DATA_PATH / "payers_clean.csv"
)

diagnoses = pd.read_csv(
    CLEANED_DATA_PATH / "diagnoses_clean.csv"
)

procedures = pd.read_csv(
    CLEANED_DATA_PATH / "procedures_clean.csv"
)

encounters = pd.read_csv(
    CLEANED_DATA_PATH / "encounters_clean.csv"
)

encounter_diagnoses = pd.read_csv(
    CLEANED_DATA_PATH / "encounter_diagnoses_clean.csv"
)

encounter_procedures = pd.read_csv(
    CLEANED_DATA_PATH / "encounter_procedures_clean.csv"
)

admissions = pd.read_csv(
    CLEANED_DATA_PATH / "admissions_clean.csv"
)

appointments = pd.read_csv(
    CLEANED_DATA_PATH / "appointments_clean.csv"
)

lab_results = pd.read_csv(
    CLEANED_DATA_PATH / "lab_results_clean.csv"
)

print(
    "All 12 cleaned business datasets loaded."
)

All 12 cleaned business datasets loaded.


### Build stg_admissions

In [91]:
stg_admissions = admissions.copy()

In [92]:
stg_admissions = stg_admissions[
    [
        "AdmissionID",
        "EncounterID",
        "PatientID",
        "DepartmentID",
        "AdmissionDateTime",
        "DischargeDateTime",
        "AdmissionType",
        "DischargeDisposition",
        "FollowUpRequiredFlag",
        "FollowUpCompletedFlag",
        "FollowUpDate",
        "LOSValidFlag",
        "LengthOfStayDays",
        "FollowUpDateValidFlag",
        "DaysToFollowUp",
        "AdmissionQualityFlag"
    ]
]

In [93]:
stg_admissions.head()

,AdmissionID,EncounterID,PatientID,DepartmentID,AdmissionDateTime,DischargeDateTime,AdmissionType,DischargeDisposition,FollowUpRequiredFlag,FollowUpCompletedFlag,FollowUpDate,LOSValidFlag,LengthOfStayDays,FollowUpDateValidFlag,DaysToFollowUp,AdmissionQualityFlag
0,ADM000001,ENC000016,PAT005222,DEP013,2024-08-03 08:14:00,2024-08-07 06:03:56.857084,Emergency,Skilled Nursing Facility,1,1,2024-09-06,1,3.909686,1,29.747259,1
1,ADM000002,ENC000024,PAT001049,DEP002,2024-10-27 21:25:00,2024-11-03 17:05:12.067328,Emergency,Home,1,0,NaN,1,6.819584,1,NaN,1
2,ADM000003,ENC000026,PAT004167,DEP002,2025-02-26 10:32:00,2025-03-05 05:48:53.130791,Elective,Home,1,0,NaN,1,6.803393,1,NaN,1
3,ADM000004,ENC000039,PAT006068,DEP002,2025-12-26 09:16:00,2025-12-28 22:47:58.253861,Elective,Home,1,1,2026-01-04,1,2.563869,1,6.050020,1
4,ADM000005,ENC000040,PAT001095,DEP013,2025-09-27 20:01:00,2025-10-04 05:23:25.600755,Emergency,Home,1,0,NaN,1,6.390574,1,NaN,1


In [94]:
print(
    "Rows:",
    len(stg_admissions)
)

print(
    "Duplicate AdmissionID:",
    stg_admissions[
        "AdmissionID"
    ].duplicated().sum()
)

print(
    "Negative LOS:",
    (
        stg_admissions[
            "LengthOfStayDays"
        ] < 0
    ).sum()
)

Rows: 17740
Duplicate AdmissionID: 0
Negative LOS: 0


### Build stg_appointments

In [95]:
stg_appointments = appointments.copy()

In [96]:
stg_appointments[
    "SourcePatientID"
] = stg_appointments[
    "PatientID"
]

stg_appointments[
    "SourceProviderID"
] = stg_appointments[
    "ProviderID"
]

stg_appointments[
    "SourceDepartmentID"
] = stg_appointments[
    "DepartmentID"
]

In [97]:
stg_appointments[
    "PatientID"
] = stg_appointments[
    "PatientIDClean"
]

stg_appointments[
    "ProviderID"
] = stg_appointments[
    "ProviderIDClean"
]

stg_appointments[
    "DepartmentID"
] = stg_appointments[
    "DepartmentIDClean"
]

In [98]:
stg_appointments[
    "SourceAppointmentStatus"
] = stg_appointments[
    "AppointmentStatus"
]

In [99]:
stg_appointments[
    "AppointmentStatus"
] = stg_appointments[
    "AppointmentStatusClean"
]

In [100]:
stg_appointments[
    "SourceCancellationReason"
] = stg_appointments[
    "CancellationReason"
]

stg_appointments[
    "CancellationReason"
] = stg_appointments[
    "CancellationReasonClean"
]

In [101]:
stg_appointments = stg_appointments.drop(
    columns=[
        "PatientIDClean",
        "ProviderIDClean",
        "DepartmentIDClean",
        "AppointmentStatusClean",
        "AppointmentStatusSource",
        "CancellationReasonClean"
    ],
    errors="ignore"
)

In [102]:
stg_appointments[
    [
        "AppointmentID",
        "PatientID",
        "ProviderID",
        "DepartmentID",
        "AppointmentStatus",
        "ScheduledDate",
        "AppointmentDateTime",
        "BookingLeadDays"
    ]
].head()

,AppointmentID,PatientID,ProviderID,DepartmentID,AppointmentStatus,ScheduledDate,AppointmentDateTime,BookingLeadDays
0,APT000001,PAT001942,PRV023,DEP025,Completed,2025-11-11,2025-11-23 16:00:00,12.0
1,APT000002,PAT009810,PRV074,DEP004,Completed,2025-08-15,2025-08-29 16:45:00,14.0
2,APT000003,PAT008784,PRV057,DEP021,Completed,2025-04-15,2025-06-10 14:00:00,56.0
3,APT000004,PAT004494,PRV106,DEP019,Cancelled,2024-09-03,2024-11-01 16:15:00,59.0
4,APT000005,PAT009138,PRV116,DEP009,Completed,2025-06-01,2025-06-04 13:45:00,3.0


In [103]:
print(
    "Duplicate AppointmentID:",
    stg_appointments[
        "AppointmentID"
    ].duplicated().sum()
)

print(
    "Negative BookingLeadDays:",
    (
        stg_appointments[
            "BookingLeadDays"
        ] < 0
    ).sum()
)

Duplicate AppointmentID: 0
Negative BookingLeadDays: 0


### Build stg_lab_results

In [104]:
stg_lab_results = lab_results.copy()

In [105]:
stg_lab_results[
    "SourcePatientID"
] = stg_lab_results[
    "PatientID"
]

stg_lab_results[
    "PatientID"
] = stg_lab_results[
    "PatientIDClean"
]

In [106]:
stg_lab_results[
    "SourceResultValue"
] = stg_lab_results[
    "ResultValue"
]

In [107]:
stg_lab_results[
    "ResultValue"
] = stg_lab_results[
    "ResultValueClean"
]

In [108]:
stg_lab_results[
    "SourceResultFlag"
] = stg_lab_results[
    "ResultFlag"
]

In [109]:
stg_lab_results[
    "ResultFlag"
] = stg_lab_results[
    "ResultFlagClean"
]

In [110]:
stg_lab_results = stg_lab_results.drop(
    columns=[
        "PatientIDClean",
        "ResultValueClean",
        "ResultFlagClean",
        "ResultFlagSource"
    ],
    errors="ignore"
)

In [111]:
stg_lab_results[
    [
        "LabResultID",
        "EncounterID",
        "PatientID",
        "LabTest",
        "SourceResultValue",
        "ResultValue",
        "ResultUnit",
        "ResultFlag",
        "StatisticalOutlierFlag",
        "ResultDateTimingStatus"
    ]
].head()

,LabResultID,EncounterID,PatientID,LabTest,SourceResultValue,ResultValue,ResultUnit,ResultFlag,StatisticalOutlierFlag,ResultDateTimingStatus
0,LAB0000001,ENC029144,PAT000376,LDL Cholesterol,137.56,137.56,mg/dL,High,0,Valid
1,LAB0000002,ENC072776,PAT003049,Glucose,95.48,95.48,mg/dL,Normal,0,Valid
2,LAB0000003,ENC006581,PAT009138,Hemoglobin,14.46,14.46,g/dL,Normal,0,Valid
3,LAB0000004,ENC089691,PAT002228,Glucose,85.53,85.53,mg/dL,Normal,0,Valid
4,LAB0000005,ENC015337,PAT008820,LDL Cholesterol,115.83,115.83,mg/dL,High,0,Valid


In [112]:
cleaned_tables = {
    "patients": patients,
    "providers": providers,
    "departments": departments,
    "payers": payers,
    "diagnoses": diagnoses,
    "procedures": procedures,
    "encounters": encounters,
    "encounter_diagnoses": encounter_diagnoses,
    "encounter_procedures": encounter_procedures,
    "admissions": admissions,
    "appointments": appointments,
    "lab_results": lab_results
}

print(
    "Cleaned tables available:",
    len(cleaned_tables)
)

Cleaned tables available: 12


### Build the complete staging dictionary

In [113]:
variables_to_check = [
    "stg_patients",
    "stg_providers",
    "stg_departments",
    "stg_payers",
    "stg_diagnoses",
    "stg_procedures",
    "stg_encounters",
    "stg_encounter_diagnoses",
    "stg_encounter_procedures",
    "stg_admissions",
    "stg_appointments",
    "stg_lab_results"
]

for variable in variables_to_check:
    print(
        variable,
        "✅ EXISTS" if variable in globals()
        else "❌ NOT CREATED"
    )

stg_patients ✅ EXISTS
stg_providers ✅ EXISTS
stg_departments ✅ EXISTS
stg_payers ✅ EXISTS
stg_diagnoses ✅ EXISTS
stg_procedures ✅ EXISTS
stg_encounters ✅ EXISTS
stg_encounter_diagnoses ✅ EXISTS
stg_encounter_procedures ✅ EXISTS
stg_admissions ✅ EXISTS
stg_appointments ✅ EXISTS
stg_lab_results ✅ EXISTS


In [114]:
# =========================================================
# RECREATE MISSING STAGING TABLES
# =========================================================

# -------------------------
# 1. PATIENTS
# -------------------------

stg_patients = patients.copy()

stg_patients = stg_patients[
    [
        "PatientID",
        "BirthDate",
        "Gender",
        "Race",
        "Ethnicity",
        "ZipCode",
        "State",
        "Region",
        "RegistrationDate",
        "BirthDateValidFlag"
    ]
]


# -------------------------
# 2-6. REFERENCE TABLES
# -------------------------

stg_providers = providers.copy()
stg_departments = departments.copy()
stg_payers = payers.copy()
stg_diagnoses = diagnoses.copy()
stg_procedures = procedures.copy()


# -------------------------
# 7. ENCOUNTERS
# -------------------------

stg_encounters = encounters.copy()

# Preserve original source values
stg_encounters["SourcePatientID"] = (
    stg_encounters["PatientID"]
)

stg_encounters["SourceProviderID"] = (
    stg_encounters["ProviderID"]
)

stg_encounters["SourceDepartmentID"] = (
    stg_encounters["DepartmentID"]
)

stg_encounters["SourceEncounterCost"] = (
    stg_encounters["EncounterCost"]
)

# Use cleaned values as canonical staging values
stg_encounters["PatientID"] = (
    stg_encounters["PatientIDClean"]
)

stg_encounters["ProviderID"] = (
    stg_encounters["ProviderIDClean"]
)

stg_encounters["DepartmentID"] = (
    stg_encounters["DepartmentIDClean"]
)

stg_encounters["EncounterCost"] = (
    stg_encounters["EncounterCostClean"]
)

stg_encounters = stg_encounters.drop(
    columns=[
        "PatientIDClean",
        "ProviderIDClean",
        "DepartmentIDClean",
        "EncounterCostClean"
    ],
    errors="ignore"
)


# -------------------------
# 8. ENCOUNTER DIAGNOSES
# -------------------------

stg_encounter_diagnoses = (
    encounter_diagnoses.copy()
)

stg_encounter_diagnoses[
    "SourceDiagnosisID"
] = stg_encounter_diagnoses[
    "DiagnosisID"
]

stg_encounter_diagnoses[
    "DiagnosisID"
] = stg_encounter_diagnoses[
    "DiagnosisIDClean"
]

stg_encounter_diagnoses = (
    stg_encounter_diagnoses.drop(
        columns=["DiagnosisIDClean"],
        errors="ignore"
    )
)


# -------------------------
# 9. ENCOUNTER PROCEDURES
# -------------------------

stg_encounter_procedures = (
    encounter_procedures.copy()
)

stg_encounter_procedures[
    "SourceProcedureID"
] = stg_encounter_procedures[
    "ProcedureID"
]

stg_encounter_procedures[
    "SourceProcedureCost"
] = stg_encounter_procedures[
    "ProcedureCost"
]

stg_encounter_procedures[
    "ProcedureID"
] = stg_encounter_procedures[
    "ProcedureIDClean"
]

stg_encounter_procedures[
    "ProcedureCost"
] = stg_encounter_procedures[
    "ProcedureCostClean"
]

stg_encounter_procedures = (
    stg_encounter_procedures.drop(
        columns=[
            "ProcedureIDClean",
            "ProcedureCostClean"
        ],
        errors="ignore"
    )
)

print("Missing staging tables recreated successfully.")

Missing staging tables recreated successfully.


In [115]:
variables_to_check = [
    "stg_patients",
    "stg_providers",
    "stg_departments",
    "stg_payers",
    "stg_diagnoses",
    "stg_procedures",
    "stg_encounters",
    "stg_encounter_diagnoses",
    "stg_encounter_procedures",
    "stg_admissions",
    "stg_appointments",
    "stg_lab_results"
]

for variable in variables_to_check:

    print(
        variable,
        "✅ EXISTS"
        if variable in globals()
        else "❌ NOT CREATED"
    )

stg_patients ✅ EXISTS
stg_providers ✅ EXISTS
stg_departments ✅ EXISTS
stg_payers ✅ EXISTS
stg_diagnoses ✅ EXISTS
stg_procedures ✅ EXISTS
stg_encounters ✅ EXISTS
stg_encounter_diagnoses ✅ EXISTS
stg_encounter_procedures ✅ EXISTS
stg_admissions ✅ EXISTS
stg_appointments ✅ EXISTS
stg_lab_results ✅ EXISTS


In [116]:
staging_tables = {
    "patients": stg_patients,
    "providers": stg_providers,
    "departments": stg_departments,
    "payers": stg_payers,
    "diagnoses": stg_diagnoses,
    "procedures": stg_procedures,
    "encounters": stg_encounters,
    "encounter_diagnoses": stg_encounter_diagnoses,
    "encounter_procedures": stg_encounter_procedures,
    "admissions": stg_admissions,
    "appointments": stg_appointments,
    "lab_results": stg_lab_results
}

print(
    "Total staging tables:",
    len(staging_tables)
)

Total staging tables: 12


### Cleaned vs Staging row counts

In [117]:
staging_reconciliation_records = []

for table_name in staging_tables.keys():

    cleaned_rows = len(
        cleaned_tables[table_name]
    )

    staging_rows = len(
        staging_tables[table_name]
    )

    staging_reconciliation_records.append({
        "Table": table_name,
        "CleanedRows": cleaned_rows,
        "StagingRows": staging_rows,
        "Difference": staging_rows - cleaned_rows
    })

staging_reconciliation_df = pd.DataFrame(
    staging_reconciliation_records
)

staging_reconciliation_df

,Table,CleanedRows,StagingRows,Difference
0,patients,9980,9980,0
1,providers,120,120,0
2,departments,25,25,0
3,payers,6,6,0
4,diagnoses,50,50,0
5,procedures,40,40,0
6,encounters,90000,90000,0
7,encounter_diagnoses,153137,153137,0
8,encounter_procedures,70419,70419,0
9,admissions,17740,17740,0


In [118]:
print(
    "Tables with row-count differences:",
    (
        staging_reconciliation_df[
            "Difference"
        ] != 0
    ).sum()
)

Tables with row-count differences: 0


### Make PayerID warehouse-safe

In [119]:
# Preserve the original payer value
if "SourcePayerID" not in stg_encounters.columns:
    stg_encounters["SourcePayerID"] = (
        stg_encounters["PayerID"]
    )

# Valid payer reference
valid_payer_mask = (
    stg_encounters["PayerID"].notna()
    &
    stg_encounters["PayerID"].isin(
        stg_payers["PayerID"]
    )
)

# Missing / invalid payer becomes UNKNOWN
stg_encounters["PayerID"] = (
    stg_encounters["PayerID"]
    .where(
        valid_payer_mask,
        "UNKNOWN"
    )
)

print(
    "Encounters mapped to UNKNOWN Payer:",
    (
        stg_encounters["PayerID"]
        == "UNKNOWN"
    ).sum()
)

Encounters mapped to UNKNOWN Payer: 0


### Add UNKNOWN dimension members

In [120]:
unknown_patient = pd.DataFrame([
    {
        "PatientID": "UNKNOWN",
        "BirthDate": pd.NaT,
        "Gender": "Unknown",
        "Race": "Unknown",
        "Ethnicity": "Unknown",
        "ZipCode": "Unknown",
        "State": "UN",
        "Region": "Unknown",
        "RegistrationDate": pd.NaT,
        "BirthDateValidFlag": 0
    }
])

if "UNKNOWN" not in stg_patients[
    "PatientID"
].values:

    stg_patients = pd.concat(
        [unknown_patient, stg_patients],
        ignore_index=True
    )

In [121]:
unknown_department = pd.DataFrame([
    {
        "DepartmentID": "UNKNOWN",
        "DepartmentName": "Unknown / Unmapped",
        "FacilityName": "Unknown / Unmapped",
        "DepartmentType": "Unknown",
        "City": "Unknown",
        "State": "UN"
    }
])

if "UNKNOWN" not in stg_departments[
    "DepartmentID"
].values:

    stg_departments = pd.concat(
        [unknown_department, stg_departments],
        ignore_index=True
    )

In [122]:
unknown_provider = pd.DataFrame([
    {
        "ProviderID": "UNKNOWN",
        "DepartmentID": "UNKNOWN",
        "ProviderType": "Unknown",
        "Specialty": "Unknown",
        "HireDate": pd.NaT,
        "ActiveFlag": 0
    }
])

if "UNKNOWN" not in stg_providers[
    "ProviderID"
].values:

    stg_providers = pd.concat(
        [unknown_provider, stg_providers],
        ignore_index=True
    )

In [123]:
unknown_payer = pd.DataFrame([
    {
        "PayerID": "UNKNOWN",
        "PayerName": "Unknown / Unmapped",
        "PayerType": "Other"
    }
])

if "UNKNOWN" not in stg_payers[
    "PayerID"
].values:

    stg_payers = pd.concat(
        [unknown_payer, stg_payers],
        ignore_index=True
    )

In [124]:
unknown_diagnosis = pd.DataFrame([
    {
        "DiagnosisID": "UNKNOWN",
        "DiagnosisCode": "UNKNOWN",
        "DiagnosisName": "Unknown / Unmapped Diagnosis",
        "DiagnosisCategory": "Other",
        "ChronicConditionFlag": 0
    }
])

if "UNKNOWN" not in stg_diagnoses[
    "DiagnosisID"
].values:

    stg_diagnoses = pd.concat(
        [unknown_diagnosis, stg_diagnoses],
        ignore_index=True
    )

In [125]:
unknown_procedure = pd.DataFrame([
    {
        "ProcedureID": "UNKNOWN",
        "ProcedureCode": "UNKNOWN",
        "ProcedureName": "Unknown / Unmapped Procedure",
        "ProcedureCategory": "Unknown",
        "StandardCost": np.nan
    }
])

if "UNKNOWN" not in stg_procedures[
    "ProcedureID"
].values:

    stg_procedures = pd.concat(
        [unknown_procedure, stg_procedures],
        ignore_index=True
    )

### Confirm exactly one UNKNOWN row

In [126]:
print(
    "UNKNOWN Patient:",
    (stg_patients["PatientID"] == "UNKNOWN").sum()
)

print(
    "UNKNOWN Provider:",
    (stg_providers["ProviderID"] == "UNKNOWN").sum()
)

print(
    "UNKNOWN Department:",
    (stg_departments["DepartmentID"] == "UNKNOWN").sum()
)

print(
    "UNKNOWN Payer:",
    (stg_payers["PayerID"] == "UNKNOWN").sum()
)

print(
    "UNKNOWN Diagnosis:",
    (stg_diagnoses["DiagnosisID"] == "UNKNOWN").sum()
)

print(
    "UNKNOWN Procedure:",
    (stg_procedures["ProcedureID"] == "UNKNOWN").sum()
)

UNKNOWN Patient: 1
UNKNOWN Provider: 1
UNKNOWN Department: 1
UNKNOWN Payer: 1
UNKNOWN Diagnosis: 1
UNKNOWN Procedure: 1


### Referential integrity validation

In [127]:
print(
    "Unresolved Encounter Patient:",
    (
        ~stg_encounters["PatientID"]
        .isin(stg_patients["PatientID"])
    ).sum()
)

print(
    "Unresolved Encounter Provider:",
    (
        ~stg_encounters["ProviderID"]
        .isin(stg_providers["ProviderID"])
    ).sum()
)

print(
    "Unresolved Encounter Department:",
    (
        ~stg_encounters["DepartmentID"]
        .isin(stg_departments["DepartmentID"])
    ).sum()
)

print(
    "Unresolved Encounter Payer:",
    (
        ~stg_encounters["PayerID"]
        .isin(stg_payers["PayerID"])
    ).sum()
)

Unresolved Encounter Patient: 0
Unresolved Encounter Provider: 0
Unresolved Encounter Department: 0
Unresolved Encounter Payer: 0


In [128]:
print(
    "Unresolved Diagnosis:",
    (
        ~stg_encounter_diagnoses[
            "DiagnosisID"
        ].isin(
            stg_diagnoses["DiagnosisID"]
        )
    ).sum()
)

print(
    "Unresolved Procedure:",
    (
        ~stg_encounter_procedures[
            "ProcedureID"
        ].isin(
            stg_procedures["ProcedureID"]
        )
    ).sum()
)

Unresolved Diagnosis: 0
Unresolved Procedure: 0


### Rebuild complete staging dictionary

In [129]:
staging_tables = {
    "patients": stg_patients,
    "providers": stg_providers,
    "departments": stg_departments,
    "payers": stg_payers,
    "diagnoses": stg_diagnoses,
    "procedures": stg_procedures,
    "encounters": stg_encounters,
    "encounter_diagnoses": stg_encounter_diagnoses,
    "encounter_procedures": stg_encounter_procedures,
    "admissions": stg_admissions,
    "appointments": stg_appointments,
    "lab_results": stg_lab_results
}

print(
    "Total staging tables:",
    len(staging_tables)
)

Total staging tables: 12


### Save all staging tables

In [130]:
for table_name, df in staging_tables.items():

    output_file = (
        STAGING_DATA_PATH
        / f"stg_{table_name}.csv"
    )

    df.to_csv(
        output_file,
        index=False
    )

    print(
        f"{output_file.name}: "
        f"{len(df):,} rows"
    )

stg_patients.csv: 9,981 rows
stg_providers.csv: 121 rows
stg_departments.csv: 26 rows
stg_payers.csv: 7 rows
stg_diagnoses.csv: 51 rows
stg_procedures.csv: 41 rows
stg_encounters.csv: 90,000 rows
stg_encounter_diagnoses.csv: 153,137 rows
stg_encounter_procedures.csv: 70,419 rows
stg_admissions.csv: 17,740 rows
stg_appointments.csv: 50,000 rows
stg_lab_results.csv: 100,000 rows


In [131]:
staging_reconciliation_df.to_csv(
    STAGING_DATA_PATH
    / "staging_reconciliation.csv",
    index=False
)

### Verify the files physically exist

In [132]:
expected_staging_files = [
    f"stg_{name}.csv"
    for name in staging_tables.keys()
]

for filename in expected_staging_files:

    path = STAGING_DATA_PATH / filename

    print(
        filename,
        "✅ EXISTS"
        if path.exists()
        else "❌ MISSING"
    )

stg_patients.csv ✅ EXISTS
stg_providers.csv ✅ EXISTS
stg_departments.csv ✅ EXISTS
stg_payers.csv ✅ EXISTS
stg_diagnoses.csv ✅ EXISTS
stg_procedures.csv ✅ EXISTS
stg_encounters.csv ✅ EXISTS
stg_encounter_diagnoses.csv ✅ EXISTS
stg_encounter_procedures.csv ✅ EXISTS
stg_admissions.csv ✅ EXISTS
stg_appointments.csv ✅ EXISTS
stg_lab_results.csv ✅ EXISTS
